# GlandSeg Automatic Script for Large WSI

## To do next:
* Write equivalent CLI '.py' script (using chatGPT4o).
* Make sure code is properly formatted.

In [1]:
##############
## INITIATE ##
##############

## Imports
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
import torch

## Import modules
from f_wsi_reader import slide_path_to_tiles_at_coordinates
from f_wsi_reader import get_tile_map
from f_wsi_reader import read_slide
from f_wsi_reader import read_region
from f_wsi_reader import get_dimensions
from f_wsi_reader import apply_all_filters
from f_wsi_reader import apply_all_transforms
from f_wsi_reader import reassemble_tiles
from f_wsi_reader import select_patches_on_grid
from f_wsi_reader import get_tissue_mask
from f_extract_objects import extract_object_pixels
from f_extract_objects import polish
from f_extract_objects import inlay_objects
from f_extract_objects import get_bounding_box_square
# from f_model import UNet as Model
from f_model_transformer import UNetTransformer as Model

#
###

In [2]:
#####################################
## FUNCTIONS FOR MODEL APPLICATION ##
#####################################

## numpy_to_device
## Goal:
## Format numpy array for model.
## Inputs:
## X (list of numpy arrays): Input array for the model (B, H, W, C).
## Outputs:
## tensors (list of torch tensors): Output tensor for model (B, C, H, W).
def numpy_to_device(X, tile_size, device):
    tensors = []
    for x in X:        
        x = cv2.resize(x, (tile_size, tile_size))
        x = torch.tensor(x)        
        x = x.swapaxes(2, 1).swapaxes(1, 0) # (H, W, C) --> (C, H, W)
        x = x.float()
        x = x[None, :, :, :] # (C, H, W) --> (B=1, C, H, W)
        x = x.to(device)
        tensors += [x]        
    return tensors

## device_to_numpy
## Goal: 
## Format tensor on device back to numpy array.
## Inputs:
## X (list of torch tensors): List of torch tensors predictions from the model (B, C, H, W).
## Outputs:
## images (list of numpy arrays): List of images produced by the model after re-formatting (B, H, W, C).
def device_to_numpy(X, tile_size):
    images = []
    for x in X:
        x = torch.sigmoid(x) * 255 # Logits to probabilities to [0, 255]
        x = x.cpu().numpy()
        x = x.swapaxes(0, 1).swapaxes(1, 2)    
        x = x.astype('uint8')
        x = cv2.resize(x, (tile_size, tile_size))
        x = np.argmax(x, axis=2).reshape(x.shape[0], x.shape[1], 1) # Only for multiple classes
        images += [x]        
    return images

#
###


In [3]:
###########
## PATHS ##
###########

## Paths
# WSI_PATHS = ['/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_37106_15_2K_HandE+-+2017-11-28+11.43.55_TissueTrain_(1.00,63062,11375,5880,10230)/reference/EU_37106_15_2K_HandE+-+2017-11-28+11.43.55_TissueTrain_(1.00,63062,11375,5880,10230).png']
# WSI_PATHS = ['/Volumes/SED/BDI/databases/RAW/datasets/PANDA/images/tiff/0a93d9a674e358ba32e4e2ec320aefc5.tiff'] # PANDA example 1
# WSI_PATHS = ['/Volumes/SED/BDI/databases/RAW/datasets/PANDA/images/tiff/0bfe4af623f6be66a82289e411320773.tiff'] # PANDA example 2
# WSI_PATHS = ['/Volumes/SED/BDI/databases/RAW/datasets/TCGA/images/svs/TCGA-EJ-5509-01A_1.svs'] # TCGA
WSI_PATHS = ['/Volumes/SED/BDI/databases/RAW/datasets/ICGCC/images/svs/TB10.0575_V2_1.svs']

#
###

In [4]:
################
## PARAMETERS ##
################

## Parameters for tile extraction
TILE_SIZE = 64 # Size of tiles for tissue extraction
FILTER_LOW = 10 # Low filter to remove black tiles
FILTER_HIGH = 240 # High filter to remove white tiles
FILTER_STD = 2 # Std filter to remove grey tiles

## Parameters for model application
LEVEL = 0 # Downsampling level 
TILE_SIZE_MODEL = 1024 # Size of tile for model
DEVICE = "mps" # Device to load model, tiles, and perform computations
PT_MODEL_IN = 'models/TransUNet_V2_0.pth' # Path to model weights
NUM_CLASSES = 3 # Number of outputs given by model
# PT_MODEL_IN = 'models/r10_3.pth' # For using UNet model instead
# NUM_CLASSES = 4 # For using UNet model instead
BATCH_SIZE = 32 # Maximum number of patches loaded at one time
DOWNSAMPLE_TILES = 4

## Parameters for gland extraction
DOWNSAMPLE_EXTRACTION = 1 # Downsample factor of mask for extraction of objects
NUM_POLISH = 1 # Number of times to erode mask (reduced clumped objects but may lose small objects)

## Parameters for storing glands
BUFFER = 32 # Added distance around bounding box (in pixels)
THRESHOLD_MIN = 100 # Minimum surface area of objects (in pixels)
THRESHOLD_MAX = 4000 * 100 # Maximum surface area of objects (in pixels)
PROP_BLACK = 0.01 # Maximum proportion of border pixels in image (proportion)
PT_OUTPUT_FOLDER = '/Users/willembonnaffe/Desktop/extracted_objects_18d' # Path where the gland images should be saved

#
###

In [5]:
################
## LOAD MODEL ##
################

## Load model
print(f"Using {DEVICE} device")
model = Model(n_classes=NUM_CLASSES).to(DEVICE)
if os.path.exists(PT_MODEL_IN):
    model.load_state_dict(torch.load(PT_MODEL_IN))
    print("Loaded model")

#
###

Using mps device
Loaded model


In [6]:
##################################
## EXTRACT GLANDS AUTOMATICALLY ##
##################################

for WSI_PATH in WSI_PATHS:
    print(WSI_PATH)    

    ##########################
    ## GET TILE COORDINATES ##
    ##########################
    
    ## Open slide
    slide = read_slide(WSI_PATH)
    slide_size = get_dimensions(slide)
    
    ## Get tile_map
    thumbnail, tile_map, canvas_size = get_tile_map(slide, TILE_SIZE) # Canvas is a space containing the WSI.
    
    ## Get tissue mask
    tile_mask = get_tissue_mask(tile_map, FILTER_LOW, FILTER_STD, FILTER_HIGH)
    
    ## Get coordinates of tiles with tissue
    tile_coordinates = (np.argwhere(tile_mask[:,:,0] > 0) * TILE_SIZE).astype(int)
    tile_coordinates = [[tile_coordinate[0], tile_coordinate[1]] for tile_coordinate in tile_coordinates] # (x: H, y: W)
    
    ## Rescale canvas size
    canvas_size_ds = canvas_size // DOWNSAMPLE_TILES
    
    #
    ###
    
    ###############################
    ## EXTRACT PATCHES FOR MODEL ##
    ###############################
    
    ## Adjust tile size based on downsampling
    ds = 1
    if LEVEL > 0: 
        ds = 4 * LEVEL
    TILE_SIZE_MODEL = int(TILE_SIZE_MODEL/ds)
    
    ## Remove patches too close to the edges
    tile_coordinates_thinned = []
    for tile_coordinates_ in tile_coordinates:
        coords_0_lo = tile_coordinates_[0] + int(TILE_SIZE/2) - int(TILE_SIZE_MODEL/2)
        coords_0_hi = tile_coordinates_[0] + int(TILE_SIZE/2) + int(TILE_SIZE_MODEL/2)
        coords_1_lo = tile_coordinates_[1] + int(TILE_SIZE/2) - int(TILE_SIZE_MODEL/2)
        coords_1_hi = tile_coordinates_[1] + int(TILE_SIZE/2) + int(TILE_SIZE_MODEL/2)
        if (coords_0_lo > 0) & (coords_0_hi < canvas_size[0]) & (coords_1_lo > 0) & (coords_1_hi < canvas_size[1]):
            tile_coordinates_thinned += [[coords_0_lo, coords_1_lo]]
    
    ## Get patches closest to grid points
    tile_coordinates_thinned = select_patches_on_grid(tile_coordinates_thinned, grid_spacing=512)
    print(f'Number of patches: {len(tile_coordinates_thinned)}')
    
    ## Rescale coordinates according to downsampling factor
    tile_coordinates_thinned_ds = [[coords[0] // DOWNSAMPLE_TILES, coords[1] // DOWNSAMPLE_TILES] for coords in tile_coordinates_thinned]
    
    #
    ###
        
    #################
    ## APPLY MODEL ##
    #################
    
    ## Apply to all tiles
    patches = []
    preds = []
    for i in range(0, len(tile_coordinates_thinned), BATCH_SIZE):
        
        ## Load patches batch
        tile_coordinates_thinned_ = tile_coordinates_thinned[i:i + BATCH_SIZE]
        patches_ = slide_path_to_tiles_at_coordinates(WSI_PATH, tile_coordinates_thinned_, TILE_SIZE_MODEL, LEVEL)
    
        ## Apply model
        X = numpy_to_device(patches_, TILE_SIZE_MODEL, DEVICE)
        with torch.no_grad():
            for x in X:                
                preds_ = model(x)        
                preds_ = device_to_numpy(preds_, TILE_SIZE_MODEL // DOWNSAMPLE_TILES)[0] # WiP: Only for single patch            
                preds += [preds_] # WiP: Only for single patch
    
        ## Reassemble downsampled WSI
        for patches__ in patches_:
            patches__ = cv2.resize(patches__, (TILE_SIZE_MODEL // DOWNSAMPLE_TILES, TILE_SIZE_MODEL // DOWNSAMPLE_TILES))
            patches += [patches__]
    
    #
    ###
    
    ####################
    ## REASSEMBLE WSI ##
    ####################
    
    ## Reassemble image
    img = reassemble_tiles(patches, tile_coordinates_thinned_ds, canvas_size_ds[0], canvas_size_ds[1])
    
    ## Compute slide-level tissue mask
    tissue_mask = cv2.resize(tile_mask, (canvas_size_ds[1], canvas_size_ds[0]), cv2.INTER_LINEAR)[:,:,None] # (H, W)
    
    ## Mask WSI
    img = img * tissue_mask
    
    #
    ###
    
    ############################
    ## REASSEMBLE PREDICTIONS ##
    ############################
    
    ## Assemble predictions
    mask = reassemble_tiles(preds, tile_coordinates_thinned_ds, canvas_size_ds[0], canvas_size_ds[1])
    mask = mask[:,:,0].reshape(mask.shape[0], mask.shape[1], 1) * tissue_mask
    
    #
    ###
    
    ########################################
    ## PREPARE MASK FOR OBJECT EXTRACTION ##
    ########################################
    
    ## Binarise mask
    mask = ((mask > 0)*1).astype('uint8')
    mask = cv2.resize(mask, (mask.shape[1] // DOWNSAMPLE_EXTRACTION, mask.shape[0] // DOWNSAMPLE_EXTRACTION))
    mask = mask[:,:,None]
    
    ## Polish mask
    for i in range(NUM_POLISH):
        mask = polish(mask)
    
    #
    ###
    
    #####################
    ## EXTRACT OBJECTS ##
    #####################
    
    ## Get objects
    gland_coords = extract_object_pixels(mask)
    print(f'Number of glands: {len(gland_coords)}')
    
    ## Rescale coordinates to original size
    rescaled_gland_coords = [coords * DOWNSAMPLE_EXTRACTION * DOWNSAMPLE_TILES for coords in gland_coords]
    rescaled_gland_coords_ds = [coords * DOWNSAMPLE_EXTRACTION for coords in gland_coords]
    
    #
    ###
    
    ####################
    ## FILTER OBJECTS ##
    ####################
    
    ## Criteria for filtering
    buffer = BUFFER // DOWNSAMPLE_TILES
    threshold_min = THRESHOLD_MIN // (DOWNSAMPLE_TILES ** 2)
    threshold_max = THRESHOLD_MAX // (DOWNSAMPLE_TILES ** 2)
    
    ## Extract and save each object
    rescaled_gland_coords_thinned = []
    for i, coords in enumerate(rescaled_gland_coords_ds):
    
        ## Discard small objects
        if (len(coords) > threshold_min) & (len(coords) < threshold_max):
            
            ## Get bounding box of the object
            x_min, y_min, x_max, y_max = get_bounding_box_square(rescaled_gland_coords_ds[i])
    
            ## Get object
            object_img = img[(x_min - buffer):(x_max + buffer), (y_min - buffer):(y_max + buffer)]
            
            ## Check size
            if (object_img.shape[0] > 0) & (object_img.shape[1] > 0):
    
                ## Check colour
                object_img = cv2.cvtColor(object_img, cv2.COLOR_BGR2RGB)
                if (object_img.mean(2) < 10).sum()/(object_img.shape[0] * object_img.shape[1]) < PROP_BLACK:    
            
                    ## Save the object coordinates
                    rescaled_gland_coords_thinned += [rescaled_gland_coords[i]]
    
    ## Check final number of glands
    print(f'Final number of glands: {len(rescaled_gland_coords_thinned)}')
    
    #
    ###
    
    ##################
    ## SAVE OBJECTS ##
    ##################
    
    ## Paths
    os.makedirs(PT_OUTPUT_FOLDER, exist_ok=True)
    
    ## Extract and save each object
    bbox_list = []
    for i, coords in enumerate(rescaled_gland_coords_thinned):
            
        ## Get bounding box of the object
        x_min, y_min, x_max, y_max = get_bounding_box_square(coords)
    
        ## Get object
        location_xy = (x_min - BUFFER, y_min - BUFFER)
        tile_size_xy = (x_max - x_min + 2 * BUFFER, y_max - y_min + 2 * BUFFER)
        object_img = read_region(slide, location_xy, LEVEL, tile_size_xy)
        object_img = cv2.cvtColor(object_img, cv2.COLOR_BGR2RGB)
           
        ## Save the object image    
        object_filename = os.path.join(PT_OUTPUT_FOLDER, f'{os.path.basename(WSI_PATH)}_{x_min}_{x_max}_{y_min}_{y_max}.png')
        cv2.imwrite(object_filename, object_img)
    
        ## Save bbox
        bbox_list += [[x_min, y_min, x_max, y_max]]
    
    ## Save bbox
    bbox_filename = os.path.join(PT_OUTPUT_FOLDER, f'{os.path.basename(WSI_PATH)}_bboxes.png')
    np.save(bbox_filename, bbox_list)
        
    #
    ###

#
###

/Volumes/SED/BDI/databases/RAW/datasets/ICGCC/images/svs/TB10.0575_V2_1.svs
Number of patches: 160


/Users/willembonnaffe/anaconda3/envs/pytorch/lib/python3.8/site-packages/torch/nn/functional.py:3959: UserWarning: MPS: passing scale factor to upsample ops is supported natively starting from macOS 13.0. Falling back on CPU. This may have performance implications. (Triggered internally at /Users/runner/work/_temp/anaconda/conda-bld/pytorch_1682343685278/work/aten/src/ATen/native/mps/operations/UpSample.mm:233.)
  return torch._C._nn.upsample_bilinear2d(input, output_size, align_corners, scale_factors)


Number of glands: 557
Final number of glands: 374
